In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pyvisa as visa
import time

kHz = 1e3
MHz = 1e6
GHz = 1e9

In [2]:
def set_V_read_I(keithley, V, I_c=1E-1):
    
    keithley.write(f"SOUR:VOLT:MODE FIXED\n SOUR:VOLT {V}")
    keithley.write(f"SENS:CURR:PROT {I_c}")
    keithley.write("TRIG:COUN 1")
    keithley.write("FORM:ELEM CURR")
    keithley.write("OUTP ON")
    keithley.write("READ?")
    I_read = keithley.read()
    
    return float(I_read)

def reset(keithley):
    keithley.write("*RST")
    set_V_read_I(keithley, 0, 0.1)

def set_frequency_GHz(srs, f):
    
    srs.write(f"FREQ{f}GHz")

def set_amp_dBm(srs, a):
    
    f = float(srs.query("FREQ? GHz"))
    if(f > 6.075):
        srs.write(f"AMPH {a}")
    else : 
        srs.write(f"AMPL {a}")



def get_frequency_GHz(srs):
    
    f = srs.query("FREQ?GHz")
    
    return float(f)

def get_amp_dBm(srs):
    
    f = float(srs.query("FREQ? GHz"))
    if(f > 6.075):
        a = srs.query("AMPH?")
    else : 
        a = srs.query("AMPL?")
    
    return float(a)

def set_modulation_off(srs):
    
    srs.write("MODL 0")

def set_pumpAmp_FlxVoltage(srs, keithley, a, V, I_c=1E-1):
    
    set_amp_dBm(srs, a)
    I_read = set_V_read_I(keithley, V, I_c=1E-1)
    a_read = get_amp_dBm(srs)
    
    return I_read, a_read

def rampdown_V(keithley, V_init, V_fin, step=0.01, t_wait=1, verbose=False):
    
    import time
    step = np.sign(V_fin - V_init)*np.abs(step)
    V_list = np.round(np.arange(V_init, V_fin + step, step),3) 
    
    for V in V_list:
        I_read = set_V_read_I(keithley, V)
        time.sleep(t_wait)
        if verbose:
            print(I_read, V)
    return I_read

In [3]:
def setupXaxis(f_center, f_span, n_points, active_channel=1):
    
    kna.write(f":SENS{active_channel}:FREQ:CENT {f_center}")
    time.sleep(0.2)
    kna.write(f":SENS{active_channel}:FREQ:SPAN {f_span}")
    time.sleep(0.2)
    kna.write(f":SENS{active_channel}:SWE:POIN {n_points}")
    time.sleep(0.2)
#     kna.write(f":SENS{active_channel}:SWE:TYPE LIN")
#     time.sleep(1)

    x_array = np.linspace(f_center - f_span/2, f_center + f_span/2, n_points)
    
    return x_array


def setupMeasurement(s_ij = "S21", meas_format = "COMP", active_channel=1):
    
    # Supply all arguments in correct units
    
    kna.write(f":CALC{active_channel}:MEAS{active_channel}:PAR {s_ij};")
    time.sleep(0.5)
    kna.write(f":CALC{active_channel}:MEAS{active_channel}:FORM {meas_format};")
    time.sleep(0.5)


def setupAveraging(n_avgs, IF_bw):
    
    kna.write(f":SENS{active_channel}:BWID {IF_bw}")
    time.sleep(0.5)
    kna.write(f":SENS{active_channel}:AVER ON")
    time.sleep(0.5)
    kna.write(f":SENS{active_channel}:AVER:COUN {n_avgs}")
    time.sleep(0.5)
    
def setPower_getRealImag(power_dBm, wait_time, active_channel=1):
    
    kna.write(f":SOUR{active_channel}:POW {power_dBm}")
    time.sleep(wait_time)
    kna.write(f":CALC{active_channel}:MEAS{active_channel}:DATA:FDAT?")
    ydata_str = kna.read()
    ydata_temp = ydata_str.split(",")
    y_data = np.array([float(d) for d in ydata_temp])
    y_data = y_data.reshape(n_points, 2)
    y_data = y_data.transpose()
    
    real, imag = y_data
    
    return real, imag

def save_data(data, fname):
    
    path = r"D:\Experiments\2023-11-17 2DRX08, Double Ring, New  Paramps Test\ParampsData\CPWPP06B\\"
    data = np.transpose(data)
    
    np.savetxt(path + fname, data)
    
def getRealImag(active_channel=1):
    
    kna.write(f":CALC{active_channel}:MEAS{active_channel}:DATA:FDAT?")
    ydata_str = kna.read()
    ydata_temp = ydata_str.split(",")
    y_data = np.array([float(d) for d in ydata_temp])
    y_data = y_data.reshape(n_points, 2)
    y_data = y_data.transpose()
    
    real, imag = y_data
    
    return real, imag

In [4]:
def collect_data():

    active_channel=1
    kna.write(f":CALC{active_channel}:MEAS{active_channel}:DATA:FDAT?")
    ydata_str = kna.read()
    ydata_temp = ydata_str.split(",")
    y_data = np.array([float(d) for d in ydata_temp])

    return y_data

In [11]:
ip_vna = "TCPIP::192.168.0.27"
ip_cs = 'GPIB::17::INSTR'
rm = visa.ResourceManager()
kna = rm.open_resource(ip_vna)
keithley = rm.open_resource(ip_cs)

In [12]:
V_init = 20
V_final = -20
V_step = -0.1
V_arr = np.arange(V_init, V_final + V_step/2, V_step)
V_arr = np.round(V_arr, 1)

In [13]:
n_points = 1501
f_center = 7 * GHz
f_span = 5 * GHz
x_array = np.linspace(f_center - f_span/2, f_center + f_span/2, n_points)

active_channel=1
n_avgs = 10
kna.write(f":SENS{active_channel}:SWE:TIME?")
sweep_time = float(kna.read())
wait_time = sweep_time*n_avgs*1.1
# save_data(data, "Transmission_AllModes.txt")

In [14]:
for V in V_arr:
    set_V_read_I(keithley, V, I_c=1E-1)
    kna.write(f":SOUR{1}:POW {-45}")
    time.sleep(0.1)
    kna.write(f":SOUR{1}:POW {-40}")
    time.sleep(wait_time)
    y_data = collect_data()
    data = [x_array, y_data]
    save_data(data, f"ParampPhaseResponse_{V}.txt")




NameError: name 'keithley2' is not defined

In [16]:
V_init, V_fin = -20, 0
rampdown_V(keithley, V_init, V_fin, step=0.1, t_wait=1, verbose=True)

-0.01001335 -20.0
-0.009963457 -19.9
-0.009913443 -19.8
-0.009863356 -19.7
-0.009813478 -19.6
-0.009763423 -19.5
-0.009713339 -19.4
-0.009663397 -19.3
-0.009613368 -19.2
-0.009563428 -19.1
-0.009513358 -19.0
-0.009463386 -18.9
-0.009413456 -18.8
-0.009363374 -18.7
-0.00931345 -18.6
-0.009263407 -18.5
-0.009213294 -18.4
-0.00916337 -18.3
-0.00911333 -18.2
-0.009063205 -18.1
-0.00901335 -18.0
-0.008963192 -17.9
-0.008913319 -17.8
-0.00886323 -17.7
-0.008813106 -17.6
-0.008763265 -17.5
-0.008713149 -17.4
-0.008662972 -17.3
-0.008613074 -17.2
-0.008562978 -17.1
-0.008513082 -17.0
-0.00846298 -16.9
-0.008412887 -16.8
-0.008362966 -16.7
-0.008312893 -16.6
-0.008262946 -16.5
-0.008212785 -16.4
-0.008162751 -16.3
-0.008112859 -16.2
-0.008062738 -16.1
-0.008012657 -16.0
-0.007962675 -15.9
-0.007912571 -15.8
-0.007862675 -15.7
-0.007812562 -15.6
-0.007762432 -15.5
-0.007712478 -15.4
-0.00766235 -15.3
-0.007612438 -15.2
-0.007562337 -15.1
-0.007512253 -15.0
-0.007462278 -14.9
-0.007412137 -14.8
-

1.456864e-08

In [41]:
for V in V_arr[:1]:
    print(V)
    kna.write(f":SOUR{1}:POW {-45}")
    time.sleep(0.1)
    kna.write(f":SOUR{1}:POW {-40}")
    time.sleep(wait_time)
    y_data = collect_data()
    data = [x_array, y_data]
    save_data(data, f"ParampPhaseResponse_{V}.txt")

20.0
